# Week 01

Exploration for this week's charts. Data is loaded through `pipeline/processing.py`, the same way the
final chart scripts load it. Charts chosen for the site get an ID and move to the pipeline.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(REPO_ROOT / "pipeline"))  # makes the pipeline modules importable here

from processing import load_interim, unique_records

pd.set_option("display.width", 200)  # keeps wide tables on one line in text output

In [ ]:
ots = load_interim()
print(ots.shape)
ots.dtypes

## Exploration

### What the data contains

One row = trade in one CN8 code, in one month, in one flow (EU/non-EU, import/export), with one partner
country, through one UK port. Measures: `Value` (pounds), `NetMass` (kg), `SuppUnit` (unit depends on the code).
Totals across minerals count the four shared codes once per mineral (see DATA.md).

In [ ]:
# How many distinct values the data covers in each dimension
ots[["mineral", "CommodityId", "CountryId", "PortId", "MonthId"]].nunique()

In [ ]:
# Total value by direction and EU/non-EU, in GBP million
(ots.groupby(["direction", "partner_group"])["Value"].sum() / 1e6).round(1)

In [ ]:
# Value by mineral, imports and exports, in GBP million
by_mineral = ots.pivot_table(index="mineral", columns="direction", values="Value", aggfunc="sum") / 1e6
by_mineral.sort_values("import", ascending=False).round(1)

In [ ]:
# Value by year and direction, in GBP million (2026 covers January to August only)
(ots.pivot_table(index="year", columns="direction", values="Value", aggfunc="sum") / 1e6).round(1)

In [ ]:
# How many of each mineral's configured codes have any trade at all
import tomllib

with open(REPO_ROOT / "config" / "minerals.toml", "rb") as f:
    minerals = tomllib.load(f)["minerals"]

codes = pd.DataFrame({
    "configured": {key: len(m["cn8"]) for key, m in minerals.items()},
    "with_trade": ots.groupby("mineral")["CommodityId"].nunique(),
})
codes["without_trade"] = codes["configured"] - codes["with_trade"]
codes.sort_values("without_trade", ascending=False)

In [ ]:
# Largest import partners across all minerals, in GBP million (real countries only)
imports = ots[(ots["direction"] == "import") & (ots["partner_type"] == "country")]
(imports.groupby("country_name")["Value"].sum().nlargest(15) / 1e6).round(1)

### Week 01 tables (prototypes)

Three tables for this week's charts, all on imports and in GBP million. Not saved yet: once settled,
each moves to a script in `pipeline/charts/` that writes it to `data/processed/`.

In [ ]:
# 1. Total UK imports of all critical and growth minerals, by month.
#    All partners included; records under shared codes counted once.
all_imports = ots[ots["direction"] == "import"]

imports_monthly = unique_records(all_imports).groupby("month", as_index=False)["Value"].sum()
imports_monthly["value_gbp_m"] = (imports_monthly["Value"] / 1e6).round(2)
imports_monthly = imports_monthly[["month", "value_gbp_m"]]

print(imports_monthly.shape)
imports_monthly.head()

In [ ]:
# 2. UK imports by partner country in 2025: value, and number of minerals supplied.
#    Real countries only (no confidential, stores or UK partners); shared-code records counted once for value.
partners_2025 = all_imports[(all_imports["year"] == 2025) & (all_imports["partner_type"] == "country")]

partner_value = unique_records(partners_2025).groupby(["country_name", "country_iso2"], as_index=False)["Value"].sum()
partner_minerals = (partners_2025[partners_2025["Value"] > 0]
                    .groupby("country_name")["mineral"].nunique()
                    .rename("minerals_supplied").reset_index())

import_partners_2025 = partner_value.merge(partner_minerals, on="country_name", how="left")
import_partners_2025["minerals_supplied"] = import_partners_2025["minerals_supplied"].fillna(0).astype(int)
import_partners_2025["value_gbp_m"] = (import_partners_2025["Value"] / 1e6).round(2)
import_partners_2025 = (import_partners_2025[["country_name", "country_iso2", "value_gbp_m", "minerals_supplied"]]
                        .sort_values("value_gbp_m", ascending=False))

print(import_partners_2025.shape)
import_partners_2025.head(15)

In [ ]:
# 3. UK imports by mineral in 2025, all partners included.
#    Each mineral's own total: records under shared codes count towards each of their minerals.
imports_by_mineral_2025 = all_imports[all_imports["year"] == 2025].groupby("mineral", as_index=False)["Value"].sum()
imports_by_mineral_2025["mineral_name"] = imports_by_mineral_2025["mineral"].map({k: m["name"] for k, m in minerals.items()})
imports_by_mineral_2025["value_gbp_m"] = (imports_by_mineral_2025["Value"] / 1e6).round(2)
imports_by_mineral_2025 = (imports_by_mineral_2025[["mineral", "mineral_name", "value_gbp_m"]]
                           .sort_values("value_gbp_m", ascending=False))

print(imports_by_mineral_2025.shape)
imports_by_mineral_2025